# Inspect the verified FineWeb cache

This notebook loads the local cache through `load_fineweb_cache`, so the completion marker, manifest, Parquet schemas, part sequence, and row counts are verified before analysis. It displays random-looking but reproducible examples, Qwen token boundaries, token-frequency estimates, and source metadata.

The cache is drawn from [Hugging Face FineWeb `sample-10BT`](https://huggingface.co/datasets/HuggingFaceFW/fineweb). Field meanings and provenance come from the [FineWeb dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb#data-fields). Sampling uses Hugging Face's [streaming buffer shuffle](https://huggingface.co/docs/datasets/stream#shuffle): shards are shuffled, then documents are randomly selected from a rolling buffer and replaced from the source stream. It is deterministic for the fixed seed and is only an approximate global shuffle.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from IPython.display import Markdown, display
from transformers import AutoTokenizer

from ciphers.kirchenbauer_et_al.src.cache_fineweb import load_fineweb_cache

CACHE_NAME = "fineweb-500k"
TOKENIZER_NAME = "Qwen/Qwen3-4B-Base"
PREVIEW_DOCUMENTS = 5
STATISTICS_DOCUMENTS = 10_000
TOKENIZATION_BATCH_SIZE = 64

/mnt/align4_drive2/adrianoh/miniconda-installation/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cache manifest and field contract

Every cached row contains: extracted `text`; Common Crawl `id`, `dump`, `url`, crawl `date`, and WARC `file_path`; detected `language` and fastText `language_score`; and `token_count` under the GPT-2 tokenizer. The Qwen token counts computed below are separate.

In [2]:
cached_dataset = load_fineweb_cache(CACHE_NAME)
cache_directory = Path(os.environ["STEGO_ARTIFACTS_DIR"]) / "datasets" / "fineweb" / CACHE_NAME
manifest = json.loads((cache_directory / "manifest.json").read_text())
display(pd.Series(manifest, name="value").to_frame())

sample_size = max(PREVIEW_DOCUMENTS, STATISTICS_DOCUMENTS)
sampled_documents = list(cached_dataset.take(sample_size))
print(f"Loaded {len(sampled_documents):,} reproducibly shuffled documents from {cache_directory}")

,value
format_version,2
cache_name,fineweb-500k
documents,500000
parts,50
part_documents,10000
source_dataset,HuggingFaceFW/fineweb
source_config,sample-10BT
source_revision,9bb295ddab0e05d785b879661af7260fed5140fc


Loaded 10,000 reproducibly shuffled documents from /mnt/align4_drive2/adrianoh/git/StegoICMLMechInterp2026/ciphers/kirchenbauer_et_al/artifacts/official/datasets/fineweb/fineweb-500k


## Example documents and provenance

These are the first examples from the deterministic shuffled stream, not the first sequential documents written to disk. Text is shown in full. URLs point to the pages from which FineWeb extracted the text, though pages may have changed or disappeared since their recorded crawl dates.

In [3]:
metadata_fields = ["id", "dump", "url", "date", "file_path", "language", "language_score", "token_count"]
for document_index, document in enumerate(sampled_documents[:PREVIEW_DOCUMENTS], start=1):
    display(Markdown(f"### Document {document_index}"))
    display(pd.Series({field: document[field] for field in metadata_fields}, name="value").to_frame())
    print(document["text"])
    print("=" * 100)

### Document 1

,value
id,<urn:uuid:c1c92d25-0e41-41e5-a007-5364b4af5c8d>
dump,CC-MAIN-2013-20
url,http://a34qxld9.livejournal.com/
date,2013-05-20T04:12:26Z
file_path,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...
language,en
language_score,0.940323
token_count,641


GO books by elsevier
The best key search books by elsevier
Cod-liver oil For the Skin - the Best and Natural Way to Hold your Sight of the Skin Healthy and Young
You among thousand people which search for a magic wand which can smooth them, bright, and a shining skin? It is good, then you should arrive a correct place. In this article we will discuss, as cod-liver oil for a skin works.
Until recently, cod-liver oil additions have been connected with warm problems, neurobehavioral disorders, an incorporated pain, the Arthritis, etc. However, from late many scientific researches have proved that cod-liver oil for a skin actually works; again because of two essential omega3 fats DHA and EPA represent in oil.
You could know that DHA and EPA - necessary omega3 fats which are necessary a body, but cannot be made a body. Therefore, to wish levels of these fats, food addition of fish is necessary. Any deficiency of these fats leads to impetuous ignition in a body which is at the bottom of occu

### Document 2

,value
id,<urn:uuid:2582d6e8-ee01-4aaf-9171-5dbfcf17c258>
dump,CC-MAIN-2013-20
url,http://twifans.com/profile/MileyRayCyrus
date,2013-05-25T07:15:08Z
file_path,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...
language,en
language_score,0.959885
token_count,88


...

,value
id,<urn:uuid:cecd2b1b-c5c6-4f9c-a39a-21910907a724>
dump,CC-MAIN-2015-14
url,http://www.gamefront.com/hi-rez-exploring-map-...
date,2015-04-02T10:29:43Z
file_path,s3://commoncrawl/crawl-data/CC-MAIN-2015-14/se...
language,en
language_score,0.965382
token_count,707


Hi-Rez ‘Exploring’ Map Editor, Tools as Tribes: Ascend Updates Slow
Official content updates for Hi-Rez Studios’ Tribes: Ascend might be all but ended, but it’s possible players will soon be able to take up the task of building new maps and other features where developers are leaving off.
Hi-Rez co-founder Tod Harris said in an interview with Game Front that the developer is potentially exploring some elements community members have been clamoring for since Tribes: Ascend was first announced: a map editor and mod tools that would allow players to create their own content.
“Really, the main community request we’re hearing, which was always a part of Tribes, is for some way for the community to create their own maps,” Harris said. “So, over the next six months, as far as what’s next for Tribes, that’s really the area we’re going to be exploring. We don’t really have any details on that, but that’s what we’re hearing mainly from the community. They’d like a way, independent of our own map

### Document 5

,value
id,<urn:uuid:4abb10cb-a1dd-4c03-b975-cfb3f5160995>
dump,CC-MAIN-2018-34
url,http://xtrenergy.ca/news/spring-cleaning
date,2018-08-14T06:26:39Z
file_path,s3://commoncrawl/crawl-data/CC-MAIN-2018-34/se...
language,en
language_score,0.926489
token_count,352


It is that time of year when the snowbanks start to melt into giant piles of gravel and rivers of muddy water flow across your forecourts. On a sunny day it almost looks to be T-Shirt weather but after stepping outside you quicikly realise it is still SUB-ZERO out. As your customers begin to wake-up from their long, cold, winter hibernation it is important to make sure your gas station meets their dreamy expectations.
Here is a list of some Spring Cleaning items to think about:
1. Maind ID is working and shows accurate price
2. Canopy / Spread-bar branding is well maintained and clean (use windex and a rag)
3. Site is well-lit at night, replace missing bulbs and ensure safety
4. Dispensers are working and pumping fuel at a normal rate (if not this could be a sign that pump calibration is needed)
5. Pick-up winter garbage from lot and surrounding area
6. Site elements are clean & tidy (dispensers, driveway, store windows...etc)
7. Windshield washer supplies are stocked and available
8. 

## Qwen token boundaries

`token_string` exposes the tokenizer's internal representation; `decoded_piece` shows how the individual token renders. Whitespace or byte-level boundary markers can therefore be distinguished from visible decoded text.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
for document_index, document in enumerate(sampled_documents[:PREVIEW_DOCUMENTS], start=1):
    token_ids = tokenizer(document["text"], add_special_tokens=False)["input_ids"]
    token_rows = [
        {
            "position": position,
            "token_id": token_id,
            "token_string": repr(tokenizer.convert_ids_to_tokens(token_id)),
            "decoded_piece": repr(tokenizer.decode([token_id])),
        }
        for position, token_id in enumerate(token_ids)
    ]
    display(Markdown(f"### Document {document_index}: {len(token_ids):,} Qwen tokens"))
    display(pd.DataFrame(token_rows))

### Document 1: 622 Qwen tokens

,position,token_id,token_string,decoded_piece
0,0,15513,'GO','GO'
1,1,6467,'Ġbooks',' books'
2,2,553,'Ġby',' by'
3,3,770,'Ġelse',' else'
4,4,46716,'vier','vier'
...,...,...,...,...
617,617,892,'Ġwhich',' which'
618,618,498,'Ġyou',' you'
619,619,342,'Ġg',' g'
620,620,82612,'rieved','rieved'


### Document 2: 88 Qwen tokens

,position,token_id,token_string,decoded_piece
0,0,56,'Y','Y'
1,1,454,'up','up'
2,2,11,"','","','"
3,3,537,'Ġnot',' not'
4,4,2167,'Ġreally',' really'
...,...,...,...,...
83,83,2973,'ators','ators'
84,84,525,'Ġare',' are'
85,85,12418,'Ġfake',' fake'
86,86,13657,'Ġanyway',' anyway'


### Document 3: 216 Qwen tokens

,position,token_id,token_string,decoded_piece
0,0,46339,'Going','Going'
1,1,5651,'Ġtab',' tab'
2,2,51096,'loid','loid'
3,3,686,'Ġwill',' will'
4,4,3581,'Ġsave',' save'
...,...,...,...,...
211,211,28,'=','='
212,212,16,'1','1'
213,213,24,'9','9'
214,214,20,'5','5'


### Document 4: 639 Qwen tokens

,position,token_id,token_string,decoded_piece
0,0,13048,'Hi','Hi'
1,1,67960,'-Re','-Re'
2,2,89,'z','z'
3,3,3369,'ĠâĢĺ',' ‘'
4,4,43953,'Expl','Expl'
...,...,...,...,...
634,634,4436,'Ġisn',' isn'
635,635,1405,'âĢĻt','’t'
636,636,17387,'Ġproducing',' producing'
637,637,432,'Ġit',' it'


### Document 5: 336 Qwen tokens

,position,token_id,token_string,decoded_piece
0,0,2132,'It','It'
1,1,374,'Ġis',' is'
2,2,429,'Ġthat',' that'
3,3,882,'Ġtime',' time'
4,4,315,'Ġof',' of'
...,...,...,...,...
331,331,264,'Ġa',' a'
332,332,23321,'Ġdealer',' dealer'
333,333,16227,'Ġdriven',' driven'
334,334,2813,'Ġcompany',' company'


## Lengths, source domains, and Common Crawl dumps

These estimates cover only `STATISTICS_DOCUMENTS` documents from the reproducibly shuffled stream. They are not exact statistics for all 500,000 cached documents.

In [5]:
statistics_documents = sampled_documents[:STATISTICS_DOCUMENTS]
qwen_token_counts = []
token_counter = Counter()
for batch_start in range(0, len(statistics_documents), TOKENIZATION_BATCH_SIZE):
    document_batch = statistics_documents[batch_start : batch_start + TOKENIZATION_BATCH_SIZE]
    encoded_batch = tokenizer(
        [document["text"] for document in document_batch],
        add_special_tokens=False,
        return_attention_mask=False,
        truncation=False,
    )["input_ids"]
    for token_ids in encoded_batch:
        qwen_token_counts.append(len(token_ids))
        token_counter.update(token_ids)

length_statistics = pd.DataFrame(
    {
        "characters": [len(document["text"]) for document in statistics_documents],
        "gpt2_tokens_from_fineweb": [document["token_count"] for document in statistics_documents],
        "qwen_tokens_computed_here": qwen_token_counts,
        "language_score": [document["language_score"] for document in statistics_documents],
    }
)
display(length_statistics.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

source_domains = Counter(urlparse(document["url"]).netloc for document in statistics_documents)
display(pd.DataFrame(source_domains.most_common(25), columns=["source_domain", "documents"]))

common_crawl_dumps = Counter(document["dump"] for document in statistics_documents)
display(pd.DataFrame(common_crawl_dumps.most_common(), columns=["common_crawl_dump", "documents"]))

,count,mean,std,min,50%,90%,95%,99%,max
characters,10000.0,3087.91390,7763.618862,195.000000,1770.500000,6183.200000,8991.350000,20505.530000,461239.000000
gpt2_tokens_from_fineweb,10000.0,698.58810,1775.938864,46.000000,400.000000,1391.100000,2035.050000,4443.070000,110533.000000
qwen_tokens_computed_here,10000.0,686.41070,1756.513501,47.000000,393.000000,1349.200000,1958.300000,4481.410000,106834.000000
language_score,10000.0,0.92776,0.056592,0.650529,0.945149,0.977747,0.983431,0.990521,0.998443


,source_domain,documents
0,www.theguardian.com,13
1,bleacherreport.com,12
2,en.wikipedia.org,11
3,www.washingtonpost.com,10
4,mail-index.netbsd.org,9
5,www.upi.com,7
6,www.hollywoodreporter.com,6
7,www.bloomberg.com,6
8,variety.com,6
9,www.ign.com,6


,common_crawl_dump,documents
0,CC-MAIN-2017-34,1617
1,CC-MAIN-2022-21,1533
2,CC-MAIN-2019-35,1364
3,CC-MAIN-2013-20,1117
4,CC-MAIN-2018-34,1115
5,CC-MAIN-2016-30,1108
6,CC-MAIN-2020-45,1095
7,CC-MAIN-2015-14,1051


## 100 most common Qwen tokens

Relative frequency is `token_count / total_Qwen_tokens_in_the_sample`. It is an estimate over the selected documents, not a probability supplied by FineWeb or the model.

In [6]:
total_qwen_tokens = sum(token_counter.values())
most_common_token_rows = [
    {
        "rank": rank,
        "token_id": token_id,
        "token_string": repr(tokenizer.convert_ids_to_tokens(token_id)),
        "decoded_piece": repr(tokenizer.decode([token_id])),
        "count": count,
        "relative_frequency": count / total_qwen_tokens,
    }
    for rank, (token_id, count) in enumerate(token_counter.most_common(100), start=1)
]
print(f"Counted {total_qwen_tokens:,} Qwen tokens across {len(statistics_documents):,} documents")
display(pd.DataFrame(most_common_token_rows).style.format({"relative_frequency": "{:.6%}"}))

Counted 6,864,107 Qwen tokens across 10,000 documents


,rank,token_id,token_string,decoded_piece,count,relative_frequency
0,1,11,"','","','",247910,3.611686%
1,2,279,'Ġthe',' the',236022,3.438495%
2,3,13,'.','.',173452,2.526942%
3,4,323,'Ġand',' and',144028,2.098277%
4,5,311,'Ġto',' to',138922,2.023890%
5,6,315,'Ġof',' of',128758,1.875816%
6,7,264,'Ġa',' a',107915,1.572164%
7,8,220,'Ġ',' ',85656,1.247883%
8,9,304,'Ġin',' in',83712,1.219561%
9,10,624,'.Ċ','.\n',73928,1.077023%
